# 作品① · 抖音数据完整分析

目标：用 pandas 分析自己 17 条作品，回答三个问题——

1. 我平时的真实水平是多少（不被爆款骗）？
2. 哪类选题（topic）表现更好？
3. 哪种开头钩子（hook）更留人？

用法：从上到下依次运行每个代码格。先看输出，再看格子上面的中文说明。

## 第 1 步：读数据，看前 5 行

`read_excel` 读表格，`head()` 看前 5 行。观察每列是什么。

In [ ]:
import pandas as pd

df = pd.read_excel('C:/Users/Administrator/Desktop/作品列表导出.xlsx')
print('数据大小：', df.shape)
df[['title', 'views', 'topic', 'hook']].head()

## 第 2 步：认识数据

17 行作品，19 列字段。topic（选题）和 hook（钩子）都已经填好，
各有 4 个固定类别——这正是为了能分组对比。

In [ ]:
print('行数', df.shape[0], '列数', df.shape[1])
print('topic 类别：', df['topic'].unique().tolist())
print('hook 类别：', df['hook'].unique().tolist())

## 第 3 步：清洗数据（很重要）

两个坑必须先处理：

1. **6 条「自见」作品播放量是 0**，是自己可见的，要排除，只留「公开」。
2. `cover_ctr` 列有个 `-` 是无效值，`to_numeric(errors='coerce')` 把它变空值。

同时算好**互动率** =（赞+分享+评论+收藏）÷ 播放量，后面反复用。

In [ ]:
print('清洗前：')
print(df['audit_status'].value_counts())

df = df[df['audit_status'] == '公开'].copy()
df['cover_ctr'] = pd.to_numeric(df['cover_ctr'], errors='coerce')
df['互动率'] = (df['likes'] + df['shares'] + df['comments'] + df['favs']) / df['views']

print('清洗后剩', len(df), '条公开作品')

## 第 4 步：真实水平——平均数 vs 中位数

有一条 298 万播放的爆款，会把「平均播放量」拉得极高。

判断平时真实水平要看**中位数**。运行看看两者差多少倍。

In [ ]:
print('平均播放量：', round(df['views'].mean()))
print('中位数播放量：', round(df['views'].median()))

df.sort_values('views', ascending=False)[['title', 'topic', 'hook', 'views']].head()

## 第 5 步：按选题 topic 分组——回答「哪类选题最好」

`groupby('topic')` 按选题类型分组，看每类的条数、平均播放、中位播放、互动率、完播率。

**先别急着下结论**——盯着「大学生活」那组，它的平均值正常吗？下一步揭晓。

In [ ]:
topic_result = df.groupby('topic').agg(
    条数=('views', 'size'),
    平均播放=('views', 'mean'),
    中位播放=('views', 'median'),
    平均互动率=('互动率', 'mean'),
    平均完播=('finish_rate', 'mean')
)
topic_result.round(3)

## 第 6 步：识破爆款制造的假象（关键）

「大学生活」组只有 1 条，就是那条 298 万爆款，所以它一组平均播放 298 万——
**这不是选题的普遍水平，是那一条的独角戏**。

做法：把爆款（播放量 > 10 万）暂时剔除，再看各选题和各钩子的真实表现。

这就是教程强调的：**爆款是最大的混杂变量，别用绝对播放量下结论**。

In [ ]:
no_hit = df[df['views'] < 100000]
print('剔除爆款后剩', len(no_hit), '条')

print('--- 按选题 ---')
print(no_hit.groupby('topic').agg(条数=('views','size'), 中位播放=('views','median'), 平均互动率=('互动率','mean')).round(3).to_string())

print('--- 按钩子 ---')
no_hit.groupby('hook').agg(条数=('views','size'), 中位播放=('views','median'), 平均完播=('finish_rate','mean')).round(3)

## 第 7 步：按钩子 hook 分组——回答「哪种开头最留人」

这次重点看**平均完播率**和**2 秒跳出率**：开头抓人，完播率高、跳出率低。

同样分两次看：全部 11 条（会被爆款带偏）和剔除爆款后。

In [ ]:
print('--- 全部 11 条（含爆款，注意情绪共鸣组被带飞）---')
print(df.groupby('hook').agg(条数=('views','size'), 平均完播=('finish_rate','mean'), 平均2s跳出=('bounce_2s','mean'), 平均互动率=('互动率','mean')).round(3).to_string())

print('--- 剔除爆款后（更可信）---')
no_hit.groupby('hook').agg(条数=('views','size'), 平均完播=('finish_rate','mean'), 平均互动率=('互动率','mean')).round(3)

## 第 8 步：涨粉靠什么

播放量高不等于涨粉多。看看涨粉排名，是不是全靠那一条爆款。

In [ ]:
df[['title', 'topic', 'hook', 'views', 'new_followers']].sort_values('new_followers', ascending=False).head()

## 第 9 步：我的观察（可直接填回计划书）

下面这段是基于真实数据的结论，复制进 Notion「作品①计划书」的第三部分。

---

**1. 数据概况**
17 条作品里 6 条是「自见」（播放量为 0），排除后能分析的是 **11 条公开作品**。

**2. 真实水平看中位数，别被爆款骗**
「#大学生」那条播放量 **2,981,390**，是第二名的 293 倍。平均播放 274,035，中位数只有 1,112，差 246 倍。

**3. 假设①：选题影响播放 —— 已验证**
- 直接分组时「大学生活」平均 298 万，但那组只有 1 条爆款，是假象。
- 剔除爆款后：**情感关系**（5 条）中位播放 5,805 最强；**自我成长**（3 条）中位 543 最弱，但平均互动率 5.9% 最高（情绪日常 5.0%、情感关系 3.2%）。
- 结论：**情感关系拉播放，自我成长互动最用心但看得人少**。

**4. 假设②：钩子影响完播 —— 已验证**
- 剔除爆款后：**故事代入**平均完播 **0.353** 最高；**金句戳心** 0.262；**认知干货** 0.221 最低。
- 看全部 11 条时，认知干货的 2 秒跳出率高达 **0.579**（最不留人），情绪共鸣最低 0.412。
- 结论：**故事开头最留人，认知干货开头最容易劝退**。

**5. 假设③：收藏率与涨粉 —— 样本不足**
「#大学生」一条带来 251 个新粉丝，其余作品涨粉最多 1 个。涨粉高度集中在爆款，无法证明是收藏驱动。

**一句话结论**：下一条作品用「情感关系」选题 +「故事代入」钩子，目标把完播率从目前的 0.29 提到 0.35 以上。

---

**已知局限**：16 条图文 + 1 条视频，结论主要适用于图文；hook 是按标题预填、需看视频开头复核；样本量小，下周同一时间再导出、攒够 8 周才能下稳定结论。